In [195]:
import numpy as np
import pandas as pd
import holidays
import requests
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
# pd.set_option("display.max_rows",None)
pd.set_option("display.max_columns",None)
pd.set_option('display.width', None)         
pd.set_option('display.max_colwidth', None)   

In [196]:
def peak_score(y_true, y_pred, tolerance=3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred) <= tolerance)

def weighted_mae(y_true, y_pred, peak_threshold=10, peak_weight=2.0):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    weights = np.where(y_true >= peak_threshold, peak_weight, 1.0)
    return np.average(np.abs(y_true - y_pred), weights=weights)

def peak_aware_score(y_true, y_pred, peak_threshold=10, peak_tolerance=4, normal_tolerance=2):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    peak_mask = y_true >= peak_threshold
    ok = np.where(
        peak_mask,
        np.abs(y_true - y_pred) <= peak_tolerance,
        np.abs(y_true - y_pred) <= normal_tolerance
    )
    return np.mean(ok)

def tolerance_accuracy(y_true, y_pred, tolerance):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred) <= tolerance)

In [197]:
df = pd.read_csv("/kaggle/input/datasets/reddyrohith/round2clean/dataflipkart.csv")

In [198]:
df.head()

,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,hour,day_of_week,month,is_weekend,violation_type_clean
0,0,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengaluru, Karnataka. Pin-560068 (India)",0,MAXI-CAB,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,0,0,9.0,Madiwala,True,No Junction,0,Monday,11,False,"['WRONG PARKING', 'PARKING NEAR ROAD CROSSING']"
1,1,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony, Doddakannelli, Bengaluru, Karnataka. Pin-560035 (India)",1,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,1,1,82.0,Bellandur,False,No Junction,22,Friday,11,False,['NO PARKING']
2,2,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengaluru South City Corporation, Bengaluru, Bangalore South, Bengaluru Urban, Karnataka, 560034, India",2,MAXI-CAB,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,0,0,9.0,Madiwala,True,No Junction,0,Monday,11,False,"['WRONG PARKING', 'PARKING IN A MAIN ROAD']"
3,3,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Bengaluru, Karnataka. Pin-560072 (India)",3,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,2,2,26.0,Byatarayanapura,True,No Junction,6,Thursday,11,False,['NO PARKING']
4,4,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560009, India",4,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,3,3,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,4,Wednesday,11,False,['NO PARKING']


In [243]:
df[(df["police_station"]=="Madiwala") & (df["month"]==11) & (df["hour"]==14)]

,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,hour,day_of_week,month,is_weekend,violation_type_clean,date,is_holiday,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code


In [199]:
india_holidays = holidays.India()

In [200]:
df['created_datetime'] = pd.to_datetime(
    df['created_datetime'],
    format='ISO8601',
    utc=True
)

In [201]:
df['date'] = df['created_datetime'].dt.date
df['is_holiday'] = df['date'].isin(india_holidays)

In [202]:
df.head()

,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,hour,day_of_week,month,is_weekend,violation_type_clean,date,is_holiday
0,0,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengaluru, Karnataka. Pin-560068 (India)",0,MAXI-CAB,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,0,0,9.0,Madiwala,True,No Junction,0,Monday,11,False,"['WRONG PARKING', 'PARKING NEAR ROAD CROSSING']",2023-11-20,False
1,1,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony, Doddakannelli, Bengaluru, Karnataka. Pin-560035 (India)",1,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,1,1,82.0,Bellandur,False,No Junction,22,Friday,11,False,['NO PARKING'],2023-11-24,False
2,2,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengaluru South City Corporation, Bengaluru, Bangalore South, Bengaluru Urban, Karnataka, 560034, India",2,MAXI-CAB,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,0,0,9.0,Madiwala,True,No Junction,0,Monday,11,False,"['WRONG PARKING', 'PARKING IN A MAIN ROAD']",2023-11-20,False
3,3,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Bengaluru, Karnataka. Pin-560072 (India)",3,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,2,2,26.0,Byatarayanapura,True,No Junction,6,Thursday,11,False,['NO PARKING'],2023-11-16,False
4,4,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560009, India",4,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,3,3,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,4,Wednesday,11,False,['NO PARKING'],2023-11-22,False


In [203]:
df["is_holiday"].value_counts()

is_holiday
False    298450
Name: count, dtype: int64

In [204]:
print(df['date'].min())
print(df['date'].max())

2023-11-09
2024-04-08


In [205]:
lat = 12.9716
lon = 77.5946

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}"
    f"&longitude={lon}"
    f"&start_date={df['date'].min()}"
    f"&end_date={df['date'].max()}"
    "&hourly=temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code"
    "&timezone=auto"
)
data = requests.get(url).json()

In [206]:
weather_df = pd.DataFrame(data["hourly"])

weather_df["datetime"] = pd.to_datetime(weather_df["time"])

weather_df["date"] = weather_df["datetime"].dt.date
weather_df["hour"] = weather_df["datetime"].dt.hour

In [207]:
weather_df.head()

,time,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code,datetime,date,hour
0,2023-11-09T00:00,20.0,0.0,62,11.4,2,2023-11-09 00:00:00,2023-11-09,0
1,2023-11-09T01:00,19.8,0.0,43,5.9,1,2023-11-09 01:00:00,2023-11-09,1
2,2023-11-09T02:00,19.6,0.0,100,6.9,3,2023-11-09 02:00:00,2023-11-09,2
3,2023-11-09T03:00,20.0,0.0,100,7.2,3,2023-11-09 03:00:00,2023-11-09,3
4,2023-11-09T04:00,19.9,0.0,100,7.2,3,2023-11-09 04:00:00,2023-11-09,4


In [208]:
weather_df = weather_df[
    [
        "date",
        "hour",
        "temperature_2m",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
        "weather_code"
    ]
]

In [209]:
df = df.merge(
    weather_df,
    on=["date", "hour"],
    how="left"
)

In [210]:
min_date = df['created_datetime'].min().normalize()
max_date = df['created_datetime'].max().normalize()

stations = df['police_station'].unique()
date_range = pd.date_range(min_date, max_date, freq='D')

full_grid = pd.MultiIndex.from_product(
    [stations, date_range, range(24)],
    names=['police_station', 'date', 'hour']
).to_frame(index=False)
full_grid['date'] = full_grid['date'].dt.date

actual_counts = df.groupby(['police_station','date','hour']).size().reset_index(name='violation_count')

count_df = full_grid.merge(actual_counts, on=['police_station','date','hour'], how='left')
count_df['violation_count'] = count_df['violation_count'].fillna(0).astype(int)

# Re-merge weather, day_of_week, is_weekend onto this complete grid
count_df = count_df.merge(weather_df, on=['date','hour'], how='left')
count_df['datetime'] = pd.to_datetime(count_df['date'].astype(str)) + pd.to_timedelta(count_df['hour'], unit='h')
count_df['day_of_week'] = count_df['datetime'].dt.day_name()
count_df['is_weekend'] = count_df['datetime'].dt.dayofweek >= 5

In [211]:
df.head(2)

,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,hour,day_of_week,month,is_weekend,violation_type_clean,date,is_holiday,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code
0,0,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengaluru, Karnataka. Pin-560068 (India)",0,MAXI-CAB,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,0,0,9.0,Madiwala,True,No Junction,0,Monday,11,False,"['WRONG PARKING', 'PARKING NEAR ROAD CROSSING']",2023-11-20,False,19.2,0.0,69,12.7,2
1,1,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony, Doddakannelli, Bengaluru, Karnataka. Pin-560035 (India)",1,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,1,1,82.0,Bellandur,False,No Junction,22,Friday,11,False,['NO PARKING'],2023-11-24,False,19.8,0.0,100,15.3,3


In [212]:
count_df.head()

,police_station,date,hour,violation_count,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code,datetime,day_of_week,is_weekend
0,Madiwala,2023-11-09,0,0,20.0,0.0,62,11.4,2,2023-11-09 00:00:00,Thursday,False
1,Madiwala,2023-11-09,1,0,19.8,0.0,43,5.9,1,2023-11-09 01:00:00,Thursday,False
2,Madiwala,2023-11-09,2,0,19.6,0.0,100,6.9,3,2023-11-09 02:00:00,Thursday,False
3,Madiwala,2023-11-09,3,0,20.0,0.0,100,7.2,3,2023-11-09 03:00:00,Thursday,False
4,Madiwala,2023-11-09,4,0,19.9,0.0,100,7.2,3,2023-11-09 04:00:00,Thursday,False


In [213]:
count_df.describe()

,hour,violation_count,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code,datetime
count,196992.000000,196992.000000,196992.000000,196992.000000,196992.000000,196992.000000,196992.000000,196992
mean,11.500000,1.515036,23.853536,0.013213,54.532621,12.564145,3.549342,2024-01-23 23:30:00
min,0.000000,0.000000,14.000000,0.000000,0.000000,0.400000,0.000000,2023-11-09 00:00:00
25%,5.750000,0.000000,19.900000,0.000000,5.000000,9.800000,0.000000,2023-12-16 23:45:00
50%,11.500000,0.000000,23.000000,0.000000,63.000000,12.700000,2.000000,2024-01-23 23:30:00
75%,17.250000,0.000000,27.500000,0.000000,99.000000,15.300000,3.000000,2024-03-01 23:15:00
max,23.000000,187.000000,37.500000,4.800000,100.000000,27.000000,63.000000,2024-04-08 23:00:00
std,6.922204,5.625361,4.938056,0.134041,42.043572,3.977652,9.773124,NaN


In [214]:
(count_df["violation_count"]==0).sum()

np.int64(152051)

In [215]:
count_df.groupby('police_station')['violation_count'].agg(['mean','median','max','min','std']).sort_values('max')

,mean,median,max,min,std
police_station,,,,,
K.S. Layout,0.226974,0.0,13,0,0.863301
Jnanabharathi,0.291393,0.0,16,0,1.062517
Thalagattapura,0.238487,0.0,16,0,1.043411
No Police Station,0.092928,0.0,16,0,0.642499
Chikkabanavara,0.215186,0.0,19,0,0.957283
Jalahalli,0.317982,0.0,21,0,1.411024
Kengeri,0.127193,0.0,21,0,0.748453
Kamakshipalya,0.392270,0.0,24,0,1.739843
Sadashivanagar,0.396107,0.0,25,0,1.659087


In [216]:
count_df.groupby(count_df['precipitation']>0)['violation_count'].mean()

precipitation
False    1.554874
True     0.524166
Name: violation_count, dtype: float64

In [217]:
count_df.groupby('is_weekend')['violation_count'].mean()

is_weekend
False    1.487183
True     1.583403
Name: violation_count, dtype: float64

In [218]:
count_df["weather_code"].value_counts()

weather_code
3     80082
0     67014
2     22302
1     19980
51     6264
53      810
55      216
63      162
61      162
Name: count, dtype: int64

In [219]:
count_df[count_df["weather_code"]==63]

,police_station,date,hour,violation_count,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code,datetime,day_of_week,is_weekend
14,Madiwala,2023-11-09,14,0,22.3,4.8,99,17.3,63,2023-11-09 14:00:00,Thursday,False
352,Madiwala,2023-11-23,16,0,21.7,3.5,100,11.8,63,2023-11-23 16:00:00,Thursday,False
375,Madiwala,2023-11-24,15,0,24.2,2.7,100,11.2,63,2023-11-24 15:00:00,Friday,False
3662,Bellandur,2023-11-09,14,0,22.3,4.8,99,17.3,63,2023-11-09 14:00:00,Thursday,False
4000,Bellandur,2023-11-23,16,0,21.7,3.5,100,11.8,63,2023-11-23 16:00:00,Thursday,False
...,...,...,...,...,...,...,...,...,...,...,...,...
190048,No Police Station,2023-11-23,16,0,21.7,3.5,100,11.8,63,2023-11-23 16:00:00,Thursday,False
190071,No Police Station,2023-11-24,15,0,24.2,2.7,100,11.2,63,2023-11-24 15:00:00,Friday,False
193358,Devanahalli Airport,2023-11-09,14,0,22.3,4.8,99,17.3,63,2023-11-09 14:00:00,Thursday,False
193696,Devanahalli Airport,2023-11-23,16,0,21.7,3.5,100,11.8,63,2023-11-23 16:00:00,Thursday,False


In [220]:
count_df["hour_sin"] = np.sin(2 * np.pi * count_df["hour"] / 24)
count_df["hour_cos"] = np.cos(2 * np.pi * count_df["hour"] / 24)

In [221]:
le_station = LabelEncoder()
count_df["police_station"] = le_station.fit_transform(count_df["police_station"])

le_day = LabelEncoder()
count_df["day_of_week"] = le_day.fit_transform(count_df["day_of_week"])

In [222]:
count_df["dow_sin"] = np.sin(2*np.pi*count_df["day_of_week"]/7)
count_df["dow_cos"] = np.cos(2*np.pi*count_df["day_of_week"]/7)

In [223]:
count_df["temp_bin"] = pd.cut(
    count_df["temperature_2m"],
    bins=10
)

count_df.groupby("temp_bin")["violation_count"].mean()

/tmp/ipykernel_58/2259106716.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  count_df.groupby("temp_bin")["violation_count"].mean()


temp_bin
(13.976, 16.35]    3.660899
(16.35, 18.7]      3.185568
(18.7, 21.05]      2.591168
(21.05, 23.4]      1.634416
(23.4, 25.75]      0.902990
(25.75, 28.1]      0.426581
(28.1, 30.45]      0.349139
(30.45, 32.8]      0.086246
(32.8, 35.15]      0.066358
(35.15, 37.5]      0.012869
Name: violation_count, dtype: float64

In [224]:
count_df["is_rainy"] = (count_df["precipitation"] > 0).astype(int)

count_df.groupby("is_rainy")["violation_count"].describe()

,count,mean,std,min,25%,50%,75%,max
is_rainy,,,,,,,,
0,189378.0,1.554874,5.692250,0.0,0.0,0.0,0.0,187.0
1,7614.0,0.524166,3.434533,0.0,0.0,0.0,0.0,103.0


In [225]:
count_df.groupby("hour")["violation_count"].agg(["mean","median","count"])

,mean,median,count
hour,,,
0,2.651316,0.0,8208
1,2.090034,0.0,8208
2,3.017788,0.0,8208
3,3.132188,0.0,8208
4,3.545565,0.0,8208
5,4.152656,1.0,8208
6,3.276072,0.0,8208
7,1.779727,0.0,8208
8,1.042398,0.0,8208


In [226]:
count_df.groupby("weather_code")["violation_count"].describe()

,count,mean,std,min,25%,50%,75%,max
weather_code,,,,,,,,
0,67014.0,1.570702,5.701505,0.0,0.0,0.0,0.0,187.0
1,19980.0,1.140841,4.862387,0.0,0.0,0.0,0.0,111.0
2,22302.0,1.395615,5.305342,0.0,0.0,0.0,0.0,119.0
3,80082.0,1.689281,5.967975,0.0,0.0,0.0,1.0,145.0
51,6264.0,0.610153,3.737131,0.0,0.0,0.0,0.0,103.0
53,810.0,0.119753,1.247685,0.0,0.0,0.0,0.0,23.0
55,216.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
61,162.0,0.413580,2.185909,0.0,0.0,0.0,0.0,24.0
63,162.0,0.030864,0.392837,0.0,0.0,0.0,0.0,5.0


In [227]:
count_df = count_df.sort_values(
    ["police_station", "date", "hour"]
)
count_df["lag_1"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(1)
)
count_df["lag_2"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(2)
)
count_df["lag_3"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(3)
)
count_df["lag_4"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(4)
)
count_df["lag_24"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(24)
)
count_df["lag_168"] = (
    count_df.groupby("police_station")["violation_count"]
    .shift(168)
)
count_df["rolling_mean_24"] = (
    count_df.groupby("police_station")["violation_count"]
    .transform(lambda x: x.shift(1).rolling(24).mean())
)
count_df["rolling_mean_168"] = (
    count_df.groupby("police_station")["violation_count"]
    .transform(lambda x: x.shift(1).rolling(168).mean())
)

In [228]:
count_df.head()

,police_station,date,hour,violation_count,temperature_2m,precipitation,cloud_cover,wind_speed_10m,weather_code,datetime,day_of_week,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,temp_bin,is_rainy,lag_1,lag_2,lag_3,lag_4,lag_24,lag_168,rolling_mean_24,rolling_mean_168
164160,0,2023-11-09,0,0,20.0,0.0,62,11.4,2,2023-11-09 00:00:00,4,False,0.000000,1.000000,-0.433884,-0.900969,"(18.7, 21.05]",0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
164161,0,2023-11-09,1,0,19.8,0.0,43,5.9,1,2023-11-09 01:00:00,4,False,0.258819,0.965926,-0.433884,-0.900969,"(18.7, 21.05]",0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
164162,0,2023-11-09,2,0,19.6,0.0,100,6.9,3,2023-11-09 02:00:00,4,False,0.500000,0.866025,-0.433884,-0.900969,"(18.7, 21.05]",0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
164163,0,2023-11-09,3,0,20.0,0.0,100,7.2,3,2023-11-09 03:00:00,4,False,0.707107,0.707107,-0.433884,-0.900969,"(18.7, 21.05]",0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
164164,0,2023-11-09,4,0,19.9,0.0,100,7.2,3,2023-11-09 04:00:00,4,False,0.866025,0.500000,-0.433884,-0.900969,"(18.7, 21.05]",0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [229]:
count_df[
    ["lag_1","lag_2","lag_3","lag_4","lag_24","lag_168",
     "rolling_mean_24","rolling_mean_168"]
].isna().sum()

lag_1                 54
lag_2                108
lag_3                162
lag_4                216
lag_24              1296
lag_168             9072
rolling_mean_24     1296
rolling_mean_168    9072
dtype: int64

In [230]:
count_df.dropna(
    subset=[
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_4",
        "lag_24",
        "lag_168",
        "rolling_mean_24",
        "rolling_mean_168"
    ]
).shape

(187920, 26)

In [231]:
count_df = count_df.sort_values('datetime')

FEATURES = [
    "police_station","is_weekend","hour_sin","hour_cos","dow_sin","dow_cos",
    "temperature_2m",
    # "lag_1","lag_2","lag_3","lag_4","lag_24","lag_168",
    # "rolling_mean_24","rolling_mean_168"
]

In [232]:
split_point = int(len(count_df) * 0.8)
train_df = count_df.iloc[:split_point]
test_df  = count_df.iloc[split_point:]

X_train, y_train = train_df[FEATURES], train_df['violation_count']
X_test,  y_test  = test_df[FEATURES],  test_df['violation_count']

In [233]:
model = XGBRegressor()
# model = RandomForestRegressor()
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [234]:
preds = model.predict(X_test)
preds = np.clip(preds,0,None)

In [235]:
mae = mean_absolute_error(y_test, preds)
ps = peak_score(y_test, preds, tolerance=5)
wmae = weighted_mae(y_test, preds, peak_threshold=10, peak_weight=2.0)
pas = peak_aware_score(y_test, preds, peak_threshold=10, peak_tolerance=5, normal_tolerance=2)

print("MAE:", mae)
print("Peak Score:", ps)
print("Weighted MAE:", wmae)
print("Peak-Aware Score:", pas)

MAE: 1.6173717975616455
Peak Score: 0.9262925454960785
Weighted MAE: 2.098120993952143
Peak-Aware Score: 0.8114419147694104


In [236]:
for i in range(5,20):
    print(f"For Tolerance {i}:-")
    acc = tolerance_accuracy(y_test, preds, tolerance=i)
    print("Tolerance Accuracy:", acc)

For Tolerance 5:-
Tolerance Accuracy: 0.9262925454960785
For Tolerance 6:-
Tolerance Accuracy: 0.9393131805375771
For Tolerance 7:-
Tolerance Accuracy: 0.9493388157059823
For Tolerance 8:-
Tolerance Accuracy: 0.9574862306149903
For Tolerance 9:-
Tolerance Accuracy: 0.9634508490063199
For Tolerance 10:-
Tolerance Accuracy: 0.9682986877839539
For Tolerance 11:-
Tolerance Accuracy: 0.9719536028833219
For Tolerance 12:-
Tolerance Accuracy: 0.9760146196603975
For Tolerance 13:-
Tolerance Accuracy: 0.9788827127592071
For Tolerance 14:-
Tolerance Accuracy: 0.9813447041803092
For Tolerance 15:-
Tolerance Accuracy: 0.9836544074722708
For Tolerance 16:-
Tolerance Accuracy: 0.9855072463768116
For Tolerance 17:-
Tolerance Accuracy: 0.9873093225716388
For Tolerance 18:-
Tolerance Accuracy: 0.9888829665727557
For Tolerance 19:-
Tolerance Accuracy: 0.9900758902510216


In [237]:
dataset = pd.DataFrame({
    "actual": y_test,
    "predicted": preds.round().astype(int)
})

In [238]:
dataset.head()

,actual,predicted
79526,0,0
170726,0,0
35750,0,0
192614,0,0
145190,0,0


In [239]:
dataset.describe()

,actual,predicted
count,39399.000000,39399.000000
mean,1.376304,1.467753
std,5.182616,3.120638
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,2.000000
max,125.000000,43.000000


In [241]:
pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

hour_cos          0.283424
police_station    0.263786
hour_sin          0.181634
is_weekend        0.096039
dow_sin           0.067976
dow_cos           0.062023
temperature_2m    0.045117
dtype: float32

In [242]:
print(preds.min())
print((preds == 0).sum())
print((y_test == 0).sum())

0.0
8272
30816


In [244]:
import joblib
joblib.dump(model, 'production_model.pkl')
joblib.dump(le_station, 'le_station.pkl')
joblib.dump(le_day, 'le_day.pkl')


['le_day.pkl']